In [175]:
# ============================================================
# Core Python
# ============================================================
from datetime import datetime, timedelta, date, time
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================
# Numerical / Data
# ============================================================
import numpy as np
import pandas as pd
import os
import duckdb
DB_PATH = r"C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb"
con = duckdb.connect(DB_PATH) 
print("Connected:", DB_PATH)

# ============================================================
# Machine Learning – sklearn
# ============================================================
# Models

# Preprocessing
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import LabelEncoder, StandardScaler

# Model selection
from sklearn.model_selection import train_test_split, GridSearchCV

# Evaluation / inspection
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
)
from sklearn.inspection import permutation_importance

# Feature selection
from sklearn.feature_selection import SelectKBest, RFE, mutual_info_classif

# ============================================================
# XGBoost
# ============================================================
import xgboost as xgb
from xgboost import XGBClassifier

# ============================================================
# Model persistence
# ============================================================
from pathlib import Path

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

from joblib import dump, load

bundle = load("models/playoff_xgb_pipeline.joblib")

pipeline = bundle["pipeline"]
FEATURE_COLS = bundle["features"]

print("✅ Loaded pipeline + feature list")


print("✅ Playoff model loaded")



# ============================================================
# Explainability
# ============================================================
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️ SHAP not installed — using permutation importance fallback.")


# extract the trained XGB model from the pipeline
xgb_model = pipeline.named_steps["model"]

# TreeExplainer is correct for XGBoost
explainer = shap.TreeExplainer(xgb_model)

# ============================================================
# Visualization
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns


# Suppress SettingWithCopyWarning
pd.options.mode.chained_assignment = None
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


Connected: C:\Users\kjchr\Desktop\hockey\new_hockey_env\API data\duckdb\nhl_data.duckdb
✅ Loaded pipeline + feature list
✅ Playoff model loaded


In [ ]:

# con.close()
# print("DuckDB connection closed.")

DuckDB connection closed.


## History Data

In [ ]:
train_df = con.execute('select* from history_data').df()
train_df = train_df[train_df["season_id"] != 20192020]
train_df = train_df[train_df["season_id"] != 20202021]

assert (
    train_df
    .groupby(["season_id", "tri_code"])["made_playoffs"]
    .nunique()
    .le(1)
    .all()
), "made_playoffs varies within a team-season"
train_df = train_df.sort_values(by=['season_id','tri_code', 'game_id'])

conference_sizes = (
    train_df.groupby(["season_id", "conference"])["tri_code"]
    .nunique()
    .reset_index()
    .rename(columns={"tri_code": "conference_team_count"})
)

train_df = train_df.merge(conference_sizes, on=["season_id", "conference"], how="left")

train_df["conf_playoff_chance_baseline"] = 8 / train_df["conference_team_count"]

train_df["streak_direction"] = train_df["streak"].map({
    "W": 1,
    "L": -1
}).fillna(0).astype(int)


train_df["streak_signed"] = (
    train_df["streak_direction"] *
    train_df["streak_count"].fillna(0).astype(int)
)


train_df["streak_signed"] = train_df["streak_signed"].clip(-7, 7)


train_df.drop(columns=["streak", "streak_direction"], inplace=True)


assert (
    train_df
    .groupby(["season_id", "tri_code", "game_date"])["streak_signed"]
    .nunique()
    .le(1)
    .all()
), "❌ streak_signed varies within a team-day"

print("✅ streak_signed feature created successfully")
def bucket_rank(rank):
    if rank <= 3:
        return "tier1_elite"
    elif rank <= 6:
        return "tier2_strong"
    elif rank <= 10:
        return "tier3_bubble"
    else:
        return "tier4_bottom"
        
train_df["rank_bucket"] = train_df["conference_rank"].apply(bucket_rank)

# =====================================================
# DROP HIGH-NOISE / HIGH-RISK FEATURES (MODEL ONLY)
# =====================================================

DROP_COLS = [

    # -------------------------------------------------
    # 1. SINGLE-GAME RAW STATS (NOISY)
    # -------------------------------------------------
    "shots_for",
    "shots_against",
    "goals_for",
    "goals_against",

    "hd_sog",
    "md_sog",
    "ld_sog",

    "hd_goals",
    "md_goals",
    "ld_goals",

    "hd_against",
    "md_against",
    "ld_against",

    "game_savePct",
    "game_GAA",

    # -------------------------------------------------
    # 2. RAW PER-GAME SPECIAL TEAMS COUNTS
    # (keep cumulative / per-game rates instead)
    # -------------------------------------------------
    "pp_gf",
    "pp_ga",
    "sh_gf",
    "sh_ga",
    "times_sh",

    # -------------------------------------------------
    # 3. REDUNDANT / DOMINANT RANK FEATURES
    # (keep rank_bucket + conference_rank only)
    # -------------------------------------------------

    "division_home_rank",
    "division_road_rank",
    "division_last_10_rank",

    "conference_home_rank",
    "conference_road_rank",
    "conference_last_10_rank",

    "league_rank",
    "league_home_rank",
    "league_road_rank",
    "league_last_10_rank",



    # -------------------------------------------------
    # 4. OPTIONAL: STRONG LATE-SEASON PROXIES
    # (uncomment if probabilities converge too fast)
    # -------------------------------------------------
    # "in_playoff_spot",
]

# =====================================================
# APPLY DROP (SAFE — ignores missing)
# =====================================================

print(f"Dropping {len(DROP_COLS)} high-risk features...")

train_df_model = train_df.drop(columns=DROP_COLS, errors="ignore")

print("Remaining feature count:", train_df_model.shape[1])

train_df.head()



## Current Data

In [182]:
test_df = con.execute('select* from current_data').df()
test_df = test_df.sort_values(by=['season_id','tri_code', 'game_id'])


In [183]:

conference_sizes = (
    test_df.groupby(["season_id", "conference"])["tri_code"]
    .nunique()
    .reset_index()
    .rename(columns={"tri_code": "conference_team_count"})
)

test_df = test_df.merge(conference_sizes, on=["season_id", "conference"], how="left")

test_df["conf_playoff_chance_baseline"] = 8 / test_df["conference_team_count"]
test_df.head()


,season_id,game_id,game_date,team_id,tri_code,team_name,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gf,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga,sh_ga_pg,times_sh,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga,pp_ga_pg,sh_gf,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,game_savePct,game_GAA,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shots_for,hd_sog,md_sog,ld_sog,hd_goals,md_goals,ld_goals,shot_pct,shots_against,hd_against,md_against,ld_against,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,division_home_rank,division_road_rank,division_last_10_rank,conference,conference_rank,conference_home_rank,conference_road_rank,conference_last_10_rank,league_rank,league_home_rank,league_road_rank,league_last_10_rank,wildcard_rank,goals_for,goals_against,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct,conference_team_count,conf_playoff_chance_baseline
0,20252026,2025020021,2025-10-09,24,ANA,Anaheim Ducks,0,<NA>,0,NaN,NaN,0.495047,1,1.0,120.0,0,0.0,0,0.0,0.00000,0.000000,0,0.0,2,2.0,1.000000,1.00000,0,0.0,240.0,0,0.0,0,0.0,1.0,0.0,0.0,9,0.0,2.0,0.0,1,0.90322,3.15697,17,11,6,17,11,6,17.0,11.000000,6.000000,87,86,1,57,60,-3,1,1,87.000000,86.000000,1.00,24,36,1,60,19,17,52,9,15,41,0.678571,0.321429,0.531250,0.468750,0.559140,0.440860,24,36,1,60,24.000000,60.000000,19,9,17,15,52,41,36,6,9,21,1,0,0,2.777778,31,5,9,17,90.322581,93.100358,1,6,9,21,1,0,0,3,5,9,17,0.027778,0.903226,93.100358,87,86,57,60,-3,-3.000000,81,0,0.000000,1,0,0,0.000000,0,0,0,0,0.000000,0,0,1.00,-2,-2.00,L,1.0,1,0,1,0,0,1,3,-2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,3,-2,Pacific,8,6,8,8,Western,15,8,16,15,27,13,31,27,9,1,3,0,0,0.00,0.000000,2.851438,-12,2.00000,16,0.5
1,20252026,2025020036,2025-10-11,24,ANA,Anaheim Ducks,1,0,1,0.000000,0.000000,0.495047,4,4.0,428.0,2,2.0,2,2.0,0.50000,0.500000,0,0.0,4,4.0,0.500000,0.50000,-2,-2.0,343.0,2,2.0,0,0.0,5.0,2.0,40.0,3,2.0,6.0,33.3,12,0.73913,6.01838,8,9,-1,25,20,5,12.5,10.000000,2.500000,85,84,1,64,58,6,2,2,86.000000,85.000000,1.00,43,44,7,75,32,19,38,28,19,36,0.533333,0.466667,0.500000,0.500000,0.513514,0.486486,67,80,8,135,33.500000,67.500000,51,37,36,34,90,77,44,16,11,20,5,3,0,15.909091,23,4,10,9,73.913043,89.822134,8,22,20,41,6,3,0,9,9,19,26,0.100000,0.833333,93.333333,172,170,121,118,3,1.500000,80,1,0.500000,1,0,1,0.500000,0,0,0,2,0.500000,82,0,4.00,-1,-0.50,W,1.0,2,1,1,0,2,8,9,-1,1,0,0,0,0,0,0,0,0,0,0,0,2,1,1,0,1,0,2,8,9,-1,Pacific,5,6,2,5,Western,12,9,7,12,24,15,15,24,6,7,6,1,1,0.50,0.247524,4.448365,1,2.00000,16,0.5
2,20252026,2025020056,2025-10-14,24,ANA,Anaheim Ducks,1,1,2,0.500000,0.333333,0.512727,6,6.0,429.0,2,2.0,2,2.

In [184]:
test_df["streak_direction"] = test_df["streak"].map({
    "W": 1,
    "L": -1
}).fillna(0).astype(int)


test_df["streak_signed"] = (
    test_df["streak_direction"] *
    test_df["streak_count"].fillna(0).astype(int)
)


test_df["streak_signed"] = test_df["streak_signed"].clip(-7, 7)


test_df.drop(columns=["streak", "streak_direction"], inplace=True)


assert (
    test_df
    .groupby(["season_id", "tri_code", "game_date"])["streak_signed"]
    .nunique()
    .le(1)
    .all()
), "❌ streak_signed varies within a team-day"

print("✅ streak_signed feature created successfully")


✅ streak_signed feature created successfully


In [185]:
def bucket_rank(rank):
    if rank <= 3:
        return "tier1_elite"
    elif rank <= 6:
        return "tier2_strong"
    elif rank <= 10:
        return "tier3_bubble"
    else:
        return "tier4_bottom"
        
test_df["rank_bucket"] = test_df["conference_rank"].apply(bucket_rank)

In [ ]:
# =====================================================
# DROP HIGH-NOISE / HIGH-RISK FEATURES (MODEL ONLY)
# =====================================================

DROP_COLS = [

    # -------------------------------------------------
    # 1. SINGLE-GAME RAW STATS (NOISY)
    # -------------------------------------------------
    "shots_for",
    "shots_against",
    "goals_for",
    "goals_against",

    "hd_sog",
    "md_sog",
    "ld_sog",

    "hd_goals",
    "md_goals",
    "ld_goals",

    "hd_against",
    "md_against",
    "ld_against",

    "game_savePct",
    "game_GAA",

    # -------------------------------------------------
    # 2. RAW PER-GAME SPECIAL TEAMS COUNTS
    # (keep cumulative / per-game rates instead)
    # -------------------------------------------------
    "pp_gf",
    "pp_ga",
    "sh_gf",
    "sh_ga",
    "times_sh",

    # -------------------------------------------------
    # 3. REDUNDANT / DOMINANT RANK FEATURES
    # (keep rank_bucket + conference_rank only)
    # -------------------------------------------------

    "division_home_rank",
    "division_road_rank",
    "division_last_10_rank",

    "conference_home_rank",
    "conference_road_rank",
    "conference_last_10_rank",

    "league_rank",
    "league_home_rank",
    "league_road_rank",
    "league_last_10_rank",



    # -------------------------------------------------
    # 4. OPTIONAL: STRONG LATE-SEASON PROXIES
    # (uncomment if probabilities converge too fast)
    # -------------------------------------------------
    # "in_playoff_spot",
]

# =====================================================
# APPLY DROP (SAFE — ignores missing)
# =====================================================

print(f"Dropping {len(DROP_COLS)} high-risk features...")

current_df = test_df.drop(columns=DROP_COLS, errors="ignore")

print("Remaining feature count:", current_df.shape[1])

current_df.head()

Dropping 30 high-risk features...
Remaining feature count: 172


,season_id,game_id,game_date,team_id,tri_code,team_name,win,wins_before,games_before,win_pct_before,sos_past,sos_future,pp_opp,pp_opp_pg,pp_toi,pp_gpg,pp_net_g,pp_net_gpg,pp_pct,pp_net_pct,sh_ga_pg,times_sh_pg,pk_net_pct,pk_pct,pk_net_g,pk_net_gpg,pk_toi_pg,pp_ga_pg,sh_gf_pg,c_pp_opps,c_pp_net_g,net_pp_pct,ppng_rank,c_pp_ga,c_times_sh,net_pk_pct,pk_rank,rush_shots_for,rush_shots_against,rush_shot_differential,c_rush_shots_for,c_rush_shots_against,c_rush_diff,rush_for_per_game,rush_against_per_game,rush_diff_per_game,shot_attempts_for,shot_attempts_against,shot_attempts_differential,corsi_for,corsi_against,corsi_diff,c_shots_diff,games_played,shot_attempts_for_per_game,shot_attempts_against_per_game,shots_diff_per_game,SCF,SCSF,SCGF,SCA,HDCF,MDCF,LDCF,HDCA,MDCA,LDCA,HDCF_pct,HDCA_pct,MDCF_pct,MDCA_pct,LDCF_pct,LDCA_pct,c_SCF,c_SCSF,c_SCGF,c_SCA,SCF_pg,SCA_pg,c_HDCF,c_HDCA,c_MDCF,c_MDCA,c_LDCF,c_LDCA,shot_pct,sv_pct,pdo,c_goals_for,c_hd_sog,c_md_sog,c_ld_sog,c_hd_goals,c_md_goals,c_ld_goals,c_goals_against,c_hd_against,c_md_against,c_ld_against,c_shot_pct,c_sv_pct,c_pdo,c_shots_for,c_shots_against,c_corsi_for,c_corsi_against,c_corsi_diff,corsi_diff_per_game,games_remaining,wins,wins_pctg,losses,ot_losses,rotw,rotw_pctg,rw,so_losses,so_wins,points,points_pct,projected_points,in_playoff_spot,gf_pctg,goal_diff,goal_diff_pctg,streak_count,last_10_gp,last_10_wins,last_10_losses,last_10_otl,last_10_points,last_10_gf,last_10_ga,last_10_goal_diff,last_10_rotw,last_10_rw,home_gp,home_wins,home_losses,home_otlosses,home_rotw,home_rw,home_points,home_gf,home_ga,home_goal_diff,road_gp,road_wins,road_losses,road_ot_losses,road_rotw,road_rw,road_points,road_gf,road_ga,road_goal_diff,division,division_rank,conference,conference_rank,wildcard_rank,ot_wins,overtimes,ot_games_pct,pt_pct_x_sos,gaa_x_sv,gd_x_rushdiff,pk_load_x_pk_pct,conference_team_count,conf_playoff_chance_baseline,streak_signed,rank_bucket
0,20252026,2025020021,2025-10-09,24,ANA,Anaheim Ducks,0,<NA>,0,NaN,NaN,0.495047,1,1.0,120.0,0.0,0,0.0,0.00000,0.000000,0.0,2.0,1.000000,1.00000,0,0.0,240.0,0.0,0.0,1.0,0.0,0.0,9,0.0,2.0,0.0,1,17,11,6,17,11,6,17.0,11.000000,6.000000,87,86,1,57,60,-3,1,1,87.000000,86.000000,1.00,24,36,1,60,19,17,52,9,15,41,0.678571,0.321429,0.531250,0.468750,0.559140,0.440860,24,36,1,60,24.000000,60.000000,19,9,17,15,52,41,2.777778,90.322581,93.100358,1,6,9,21,1,0,0,3,5,9,17,0.027778,0.903226,93.100358,87,86,57,60,-3,-3.000000,81,0,0.000000,1,0,0,0.000000,0,0,0,0,0.000000,0,0,1.00,-2,-2.00,1.0,1,0,1,0,0,1,3,-2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,3,-2,Pacific,8,Western,15,9,0,0,0.00,0.000000,2.851438,-12,2.00000,16,0.5,-1,tier4_bottom
1,20252026,2025020036,2025-10-11,24,ANA,Anaheim Ducks,1,0,1,0.000000,0.000000,0.495047,4,4.0,428.0,2.0,2,2.0,0.50000,0.500000,0.0,4.0,0.500000,0.50000,-2,-2.0,343.0,2.0,0.0,5.0,2.0,40.0,3,2.0,6.0,33.3,12,8,9,-1,25,20,5,12.5,10.000000,2.500000,85,84,1,64,58,6,2,2,86.000000,85.000000,1.00,43,44,7,75,32,19,38,28,19,36,0.533333,0.466667,0.500000,0.500000,0.513514,0.486486,67,80,8,135,33.500000,67.500000,51,37,36,34,90,77,15.909091,73.913043,89.822134,8,22,20,41,6,3,0,9,9,19,26,0.100000,0.833333,93.333333,172,170,121,118,3,1.500000,80,1,0.500000,1,0,1,0.500000,0,0,0,2,0.500000,82,0,4.00,-1,-0.50,1.0,2,1,1,0,2,8,9,-1,1,0,0,0,0,0,0,0,0,0,0,0,2,1,1,0,1,0,2,8,9,-1,Pacific,5,Western,12,6,1,1,0.50,0.247524,4.448365,1,2.00000,16,0.5,1,tier4_bottom
2,20252026,2025020056,2025-10-14,24,ANA,Anaheim Ducks,1,1,2,0.500000,0.333333,0.512727,6,6.0,429.0,2.0,2,2.0,0.33333,0.333333,0.0,1.0,1.000000,1.00000,0,0.0,120.0,0.0,0.0,11.0,4.0,36.4,2,2.0,7.0,28.6,14,11,9,2,36,29,7,12.0,9.666667,2.333333,69,68,1,43,44,-1,3,3,80.333333,79.333333,1.00,37,22,4,73,13,29,29,15,26,23,0.464286,0.535714,0.527273,0.472727,0.557692,0.442308,104,102,12,208,34.666667,69.333333,64,52,65,60,119,100,18.181818,88.000000,106.181818,12,25,29,51,7,4,2,12,20,26,34,0.117647,0.848101,96.574832,241,238,164,162,2,0.666667,79,2,0.666667,1,0,2,0.666667,1,0,0,4,0.666667,109,0,4.00,0,0.00

In [187]:
# LEAK_COLS = [
#     "game_id", "game_date", "date",
#     "clinch_type", "made_playoffs",
#     "team_name", "opponent",
#     "season_id"
# ]

# ID_COLS = [
#     "team_id", "opponent_id", "tri_code"
# ]

# # 
# # Build features EXACTLY like training
# X_current = (
#     current_df
#     .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
#     .select_dtypes(include="number")
#     .fillna(0)
# )

# X_current = pd.concat([
#     X_current,
#     pd.get_dummies(current_df["rank_bucket"], prefix="rank"),
#     pd.get_dummies(current_df["conference"], prefix="conf")
# ], axis=1)

# # Hard alignment
# X_current = X_current.reindex(columns=FEATURE_COLS, fill_value=0)

# # 🔥 Call the persisted model
# current_df["playoff_prob"] = pipeline.predict_proba(X_current)[:, 1]
# current_df["playoff_prob_row"] = pipeline.predict_proba(X_current)[:, 1]
# print("✅ Row-level model probabilities added")

# latest_snapshot = (
#     current_df
#     .sort_values(["team_id", "game_date"])
#     .groupby("team_id", as_index=False)
#     .tail(1)
# )

# print("✅ Latest snapshot per team:", latest_snapshot.shape)

# latest_snapshot["playoff_prob_team"] = (
#     latest_snapshot["playoff_prob_row"] /
#     latest_snapshot["playoff_prob_row"].sum()
# ) * 16

# latest_snapshot["playoff_prob_team"] = latest_snapshot["playoff_prob_team"].clip(0, 1)
# def season_bucket(gp):
#     if gp <= 20:
#         return "early"
#     elif gp <= 50:
#         return "mid"
#     else:
#         return "late"

# latest_snapshot["season_bucket"] = latest_snapshot["games_played"].apply(season_bucket)

# def adjust_for_uncertainty(p, bucket):
#     if bucket == "early":
#         return 0.45 * p + 0.275
#     return p

# latest_snapshot["playoff_prob_display"] = latest_snapshot.apply(
#     lambda r: adjust_for_uncertainty(r["playoff_prob_team"], r["season_bucket"]),
#     axis=1
# )

# def prob_tier(p):
#     if p >= 0.85:
#         return "Lock"
#     elif p >= 0.65:
#         return "Likely"
#     elif p >= 0.45:
#         return "Bubble"
#     elif p >= 0.25:
#         return "Danger"
#     else:
#         return "Long Shot"

# latest_snapshot["playoff_tier"] = latest_snapshot["playoff_prob_display"].apply(prob_tier)

# latest_snapshot.head()

# Step 1: Train Model
Given all snapshots, what features generally matter?

In [188]:
def add_wc_field_pressure_features(df):
    """
    Add wild-card cutline and field-pressure features
    that respect NHL playoff structure:
      - Top 3 per division
      - Next 2 wild cards per conference
    """

    out = df.copy()

    # -------------------------------------------------
    # 1. Compute WC-eligible teams
    # -------------------------------------------------
    wc_eligible = out[out["division_rank"] > 3].copy()

    wc_cutline = (
        wc_eligible
        .sort_values(
            ["season_id", "game_date", "conference", "points", "points_pct"],
            ascending=[True, True, True, False, False]
        )
        .groupby(
            ["season_id", "game_date", "conference"],
            as_index=False
        )
        .nth(1)  # 2nd wild card
        .reset_index(drop=True)
        [["season_id", "game_date", "conference", "points", "points_pct"]]
        .rename(columns={
            "points": "wc2_points",
            "points_pct": "wc2_points_pct"
        })
    )

    # -------------------------------------------------
    # 2. Merge WC cutline back
    # -------------------------------------------------
    out = out.merge(
        wc_cutline,
        on=["season_id", "game_date", "conference"],
        how="left"
    )

    # -------------------------------------------------
    # 3. Margin to WC cutline
    # -------------------------------------------------
    out["wc_points_margin"] = out["points"] - out["wc2_points"]
    out["wc_points_pct_margin"] = out["points_pct"] - out["wc2_points_pct"]

    # -------------------------------------------------
    # 4. Binary status flags
    # -------------------------------------------------
    out["in_wc_spot"] = (out["wildcard_rank"] <= 2).astype(int)
    out["in_div_top3"] = (out["division_rank"] <= 3).astype(int)

    # -------------------------------------------------
    # 5. Bubble indicator
    # -------------------------------------------------
    out["on_wc_bubble"] = (
        out["wildcard_rank"].between(1, 4)
    ).astype(int)

    return out


train_df = add_wc_field_pressure_features(train_df)
test_df = add_wc_field_pressure_features(test_df)



In [189]:
def add_wc_density_features(df, windows=(2, 4, 6)):
    """
    Add wild-card crowding features:
    number of WC-eligible teams within X points of the WC cutline.
    """

    out = df.copy()

    # Only WC-eligible teams create WC pressure
    wc_mask = out["division_rank"] > 3

    for w in windows:
        col = f"wc_teams_within_{w}pts"

        out[col] = (
            out
            .assign(
                diff=lambda x: (x["points"] - x["wc2_points"]).abs()
            )
            .where(wc_mask)
            .groupby(["season_id", "game_date", "conference"])["diff"]
            .transform(lambda s: (s <= w).sum())
            .fillna(0)
            .astype(int)
        )

    return out

train_df = add_wc_density_features(train_df)
test_df = add_wc_density_features(test_df)


In [190]:
# baseline_probs = latest_snapshot[[
#     "team_id",
#     "playoff_prob_row",
#     "playoff_prob_team"
# ]].copy()


In [191]:
new_features = [
    "wc_points_margin",
    "wc_points_pct_margin",
    "in_wc_spot",
    "in_div_top3",
    "on_wc_bubble",
    "wc_teams_within_2pts",
    "wc_teams_within_4pts",
    "wc_teams_within_6pts",
]

missing = [c for c in new_features if c not in train_df.columns]
assert not missing, f"❌ Missing new features: {missing}"

print("✅ Field-pressure features present")


✅ Field-pressure features present


##### RF and XGB Model

In [192]:
# =====================================================
# 1. SORT (SANITY)
# =====================================================

train_df_model = (
    train_df_model
    .sort_values(["season_id", "team_id", "game_date"])
    .reset_index(drop=True)
)

# =====================================================
# 2. DEFINE LEAKAGE + ID COLUMNS
# =====================================================

LEAK_COLS = [
    "game_id", "game_date", "date",
    "clinch_type", "made_playoffs",
    "team_name", "opponent",
    "season_id"
]

ID_COLS = [
    "team_id", "opponent_id", "tri_code"
]

# =====================================================
# 3. BUILD FEATURE MATRIX (SINGLE SOURCE OF TRUTH)
# =====================================================

X = (
    train_df_model
    .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
    .select_dtypes(include="number")
)

X = X.loc[:, X.isna().mean() < 0.20]
X = X.fillna(0)

y = train_df_model["made_playoffs"]

assert y.notna().all(), "❌ Target contains NaN values"

# =====================================================
# 4. CATEGORICAL DUMMIES (LOCKED)
# =====================================================

rank_categories = [
    "tier1_elite",
    "tier2_strong",
    "tier3_bubble",
    "tier4_bottom"
]

train_df_model["rank_bucket"] = pd.Categorical(
    train_df_model["rank_bucket"],
    categories=rank_categories,
    ordered=True
)

rank_dummies = pd.get_dummies(
    train_df_model["rank_bucket"], prefix="rank"
)
conf_dummies = pd.get_dummies(
    train_df_model["conference"], prefix="conf"
)

X = pd.concat([X, rank_dummies, conf_dummies], axis=1)

# 🔒 CANONICAL FEATURE LIST
FEATURE_COLS = X.columns.tolist()

# =====================================================
# 5. TRAIN / TEST SPLIT (ROW-LEVEL, DAILY SNAPSHOTS)
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =====================================================
# 6. XGBOOST PIPELINE (FINAL MODEL)
# =====================================================

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", XGBClassifier(
            n_estimators=500,
            max_depth=5,
            learning_rate=0.01,
            subsample=0.8,
            colsample_bytree=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
        )),
    ]
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("\n✅ XGBOOST PIPELINE REPORT\n")
print(classification_report(y_test, y_pred))

# =====================================================
# 7. SAVE MODEL
# =====================================================

import os

os.makedirs("models", exist_ok=True)

dump(
    {
        "pipeline": pipeline,
        "features": FEATURE_COLS,
        "trained_on": pd.Timestamp.utcnow(),
        "model_type": "XGB-playoff-pipeline",
        "snapshot": "daily",
    },
    "models/playoff_xgb_pipeline.joblib"
)

print("💾 Model saved: models/playoff_xgb_pipeline.joblib")

# =====================================================
# 8. BUILD DAILY PREDICTION MATRIX
# =====================================================

X_pred = (
    train_df_model
    .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
    .select_dtypes(include="number")
    .fillna(0)
)

rank_dummies = pd.get_dummies(
    train_df_model["rank_bucket"], prefix="rank"
)
conf_dummies = pd.get_dummies(
    train_df_model["conference"], prefix="conf"
)

X_pred = pd.concat([X_pred, rank_dummies, conf_dummies], axis=1)

X_pred = X_pred.reindex(columns=FEATURE_COLS, fill_value=0)

assert list(X_pred.columns) == FEATURE_COLS
print("✅ Feature alignment locked")

# =====================================================
# 9. DAILY PLAYOFF PROBABILITIES
# =====================================================

train_df_model["playoff_prob_final"] = (
    pipeline.predict_proba(X_pred)[:, 1]
)

# =====================================================
# 10. OPTIONAL TIERS
# =====================================================

def prob_tier(p):
    if p >= 0.85:
        return "Lock"
    elif p >= 0.65:
        return "Likely"
    elif p >= 0.45:
        return "Bubble"
    elif p >= 0.25:
        return "Danger"
    else:
        return "Long Shot"

train_df_model["playoff_tier"] = (
    train_df_model["playoff_prob_final"].apply(prob_tier)
)

print("\n✅ COMPLETE — DAILY GAME-BY-GAME PLAYOFF ODDS READY")



✅ XGBOOST PIPELINE REPORT

              precision    recall  f1-score   support

         0.0       0.92      0.92      0.92      2150
         1.0       0.92      0.93      0.93      2308

    accuracy                           0.92      4458
   macro avg       0.92      0.92      0.92      4458
weighted avg       0.92      0.92      0.92      4458

💾 Model saved: models/playoff_xgb_pipeline.joblib
✅ Feature alignment locked

✅ COMPLETE — DAILY GAME-BY-GAME PLAYOFF ODDS READY


In [193]:
from sklearn.metrics import brier_score_loss

print(
    "Brier score:",
    brier_score_loss(y_test, y_prob)
)


Brier score: 0.058353910354696635


In [194]:
train_df_model["playoff_prob_final"].describe()


count    22290.000000
mean         0.517742
std          0.397059
min          0.003788
25%          0.084598
50%          0.555069
75%          0.938531
max          0.995765
Name: playoff_prob_final, dtype: float64

# Step 2: Introducing Games Played
Given where we are in the season, what features matter right now?
- The model stays frozen
- We slice the data by games_played
- We measure feature importance within each slice

#### STEP 2A: Make games_played a first-class season clock

#### STEP 2B: Permutation importance by season bucket
For each bucket:
1. Filter rows in that bucket
2. Measure importance using the frozen model
3. Store and compare results

In [ ]:
# =====================================================
# SEASON PHASE BUCKETS
# =====================================================

def season_bucket(gp):
    if gp <= 20:
        return "early"
    elif gp <= 50:
        return "mid"
    else:
        return "late"

train_df_model["season_bucket"] = (
    train_df_model["games_played"].apply(season_bucket)
)

# =====================================================
# PERMUTATION IMPORTANCE BY SEASON PHASE (PIPELINE)
# =====================================================

bucket_importances = {}

for bucket in ["early", "mid", "late"]:
    mask = train_df_model["season_bucket"] == bucket

    X_bucket = X.loc[mask]
    y_bucket = y.loc[mask]

    # Safety check
    if len(X_bucket) < 200:
        print(f"⚠️ Skipping {bucket}: insufficient samples")
        continue

    perm = permutation_importance(
        pipeline,              # ✅ PIPELINE, not xgb_best
        X_bucket,
        y_bucket,
        n_repeats=10,
        random_state=42,
        n_jobs=-1,
        scoring="roc_auc",     # recommended for prob models
    )

    bucket_importances[bucket] = (
        pd.Series(
            perm.importances_mean,
            index=FEATURE_COLS
        )
        .sort_values(ascending=False)
    )

    print(f"✅ Permutation importance computed for: {bucket}")


In [ ]:
for bucket, imp in bucket_importances.items():
    print(f"\n🔹 {bucket.upper()} SEASON")
    print(imp.head(10))


In [ ]:
#store for re-use
importance_df = (
    pd.concat(bucket_importances, axis=1)
    .rename_axis("feature")
    .reset_index()
)
importance_df.head()

# Predict

In [ ]:
# =====================================================
# 0. LOAD SAVED MODEL + FEATURES
# =====================================================

from joblib import load

bundle = load("models/playoff_xgb_pipeline.joblib")

pipeline = bundle["pipeline"]
FEATURE_COLS = bundle["features"]

print("✅ Loaded playoff pipeline model")

# =====================================================
# 1. SORT (CRITICAL)
# =====================================================

current_df = (
    current_df
    .sort_values(["season_id", "team_id", "game_date"])
    .reset_index(drop=True)
)

# =====================================================
# 2. BUILD FEATURE MATRIX (MUST MATCH TRAINING)
# =====================================================

X_current = (
    current_df
    .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
    .select_dtypes(include="number")
    .fillna(0)
)

# categorical dummies (LOCKED TO TRAINING)
rank_dummies = pd.get_dummies(current_df["rank_bucket"], prefix="rank")
conf_dummies = pd.get_dummies(current_df["conference"], prefix="conf")

X_current = pd.concat([X_current, rank_dummies, conf_dummies], axis=1)

# =====================================================
# 3. HARD FEATURE ALIGNMENT (NO DRIFT)
# =====================================================

X_current = X_current.reindex(
    columns=FEATURE_COLS,
    fill_value=0
)

assert list(X_current.columns) == FEATURE_COLS
print("✅ Current season feature alignment locked:", X_current.shape)

# =====================================================
# 4. MODEL OUTPUT (ROW-LEVEL SIGNAL)
# =====================================================

current_df["playoff_prob_row"] = (
    pipeline.predict_proba(X_current)[:, 1]
)

# =====================================================
# 5. COLLAPSE TO ONE ROW PER TEAM (LATEST SNAPSHOT)
# =====================================================

latest_snapshot = (
    current_df
    .sort_values(["team_id", "game_date"])
    .groupby("team_id", as_index=False)
    .tail(1)
)

assert latest_snapshot["team_id"].nunique() == latest_snapshot.shape[0]
print("✅ Latest snapshot per team:", latest_snapshot.shape)

# =====================================================
# 6. NORMALIZE TO TRUE PLAYOFF ODDS (16 SLOTS)
# =====================================================

latest_snapshot["playoff_prob_team"] = (
    latest_snapshot["playoff_prob_row"] /
    latest_snapshot["playoff_prob_row"].sum()
) * 16

# safety
latest_snapshot["playoff_prob_team"] = (
    latest_snapshot["playoff_prob_team"].clip(0, 1)
)

# =====================================================
# 7. GAMES-PLAYED BUCKET (ANALYSIS / DISPLAY)
# =====================================================

def season_bucket(gp):
    if gp <= 20:
        return "early"
    elif gp <= 50:
        return "mid"
    else:
        return "late"

latest_snapshot["season_bucket"] = (
    latest_snapshot["games_played"].apply(season_bucket)
)

# =====================================================
# 8. EARLY-SEASON UNCERTAINTY COMPRESSION
#    (APPLIED AFTER NORMALIZATION)
# =====================================================

def adjust_for_uncertainty(p, bucket):
    if bucket == "early":
        # stronger pull toward 50%
        return 0.45 * p + 0.275
    return p

latest_snapshot["playoff_prob_display"] = (
    latest_snapshot
    .apply(
        lambda r: adjust_for_uncertainty(
            r["playoff_prob_team"],
            r["season_bucket"]
        ),
        axis=1
    )
)

# =====================================================
# 9. PLAYOFF TIERS (DISPLAY ONLY)
# =====================================================

def prob_tier(p):
    if p >= 0.85:
        return "Lock"
    elif p >= 0.65:
        return "Likely"
    elif p >= 0.45:
        return "Bubble"
    elif p >= 0.25:
        return "Danger"
    else:
        return "Long Shot"

latest_snapshot["playoff_tier"] = (
    latest_snapshot["playoff_prob_display"].apply(prob_tier)
)

# =====================================================
# 10. SANITY CHECKS
# =====================================================

print("\n✅ Mean playoff probability by conference rank:")
print(
    latest_snapshot
    .groupby("conference_rank")["playoff_prob_team"]
    .mean()
    .sort_index()
)

print("\n✅ Distribution of playoff probabilities:")
print(latest_snapshot["playoff_prob_team"].describe())

# =====================================================
# 11. FINAL OUTPUT (TEAM-LEVEL)
# =====================================================

final_cols = [
    "season_id",
    "game_date",
    "team_id",
    "team_name",
    "conference",
    "conference_rank",
    "games_played",
    "season_bucket",
    "points",
    "points_pct",
    "playoff_prob_team",      # true odds (normalized)
    "playoff_prob_display",   # presentation-safe
    "playoff_tier"
 ]

current_playoff_odds = latest_snapshot[final_cols].copy()

print("\n✅ CURRENT SEASON PLAYOFF ODDS READY (FINAL)")


# Lock in Model

In [201]:
import os
import json
import pandas as pd
from datetime import datetime

from joblib import dump, load

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, brier_score_loss

from xgboost import XGBClassifier
MODEL_VERSION = "wc_field_pressure_v1"

MODEL_PATH = f"models/playoff_xgb_pipeline_{MODEL_VERSION}.joblib"
FEATURE_PATH = f"models/playoff_features_{MODEL_VERSION}.json"

LEAK_COLS = [
    "game_id", "game_date", "date",
    "clinch_type", "made_playoffs",
    "team_name", "opponent",
    "season_id"
]

ID_COLS = [
    "team_id", "opponent_id", "tri_code"
]

# =====================================================
# 4. SAVE LOCKED ARTIFACTS
# =====================================================

def save_model_bundle(pipeline, feature_cols):
    os.makedirs("models", exist_ok=True)

    dump(
        {
            "pipeline": pipeline,
            "features": feature_cols,
            "version": MODEL_VERSION,
            "trained_on": datetime.utcnow().isoformat(),
            "notes": "WC cutline + density features; race-aware calibration",
        },
        MODEL_PATH
    )

    with open(FEATURE_PATH, "w") as f:
        json.dump(feature_cols, f, indent=2)

    print(f"\n💾 Model saved → {MODEL_PATH}")
    print(f"📐 Features saved → {FEATURE_PATH}")

# =====================================================
# 5. LOAD + SCORE (INFERENCE ENTRY POINT)
# =====================================================

def load_and_score(current_df):
    """
    Canonical inference path for v1 model.
    """

    bundle = load(MODEL_PATH)
    pipeline = bundle["pipeline"]
    feature_cols = bundle["features"]

    df = add_wc_field_pressure_features(current_df)
    df = add_wc_density_features(df)

    X = (
        df
        .drop(columns=LEAK_COLS + ID_COLS, errors="ignore")
        .select_dtypes(include="number")
        .fillna(0)
        .reindex(columns=feature_cols, fill_value=0)
    )

    df["playoff_prob_row"] = pipeline.predict_proba(X)[:, 1]

    return df

## The first time running:
# pipeline, feature_cols = train_playoff_model(train_df_model)
# save_model_bundle(pipeline, feature_cols)

In [202]:
## run this after the first time always
from playoff_model_wc_pressure_v1 import load_and_score

current_df = load_and_score(current_df)
current_df.head()

ModuleNotFoundError: No module named 'playoff_model_wc_pressure_v1'

In [199]:
daily_results = current_df[['season_id', 'game_id', 'game_date', 'team_id', 'tri_code', 'team_name','games_played', 'playoff_prob_rf_cal', 'playoff_prob_xgb_cal', 'playoff_prob_ensemble', 'season_bucket', 'playoff_tier', 'rf_minus_xgb', 'playoff_prob_final', 'playoff_prob_rf', 'playoff_prob_display', 'playoff_prob_row', 'playoff_prob_team' ]]

KeyError: "['playoff_prob_rf_cal', 'playoff_prob_xgb_cal', 'playoff_prob_ensemble', 'season_bucket', 'playoff_tier', 'rf_minus_xgb', 'playoff_prob_final', 'playoff_prob_rf', 'playoff_prob_display', 'playoff_prob_row', 'playoff_prob_team'] not in index"

In [ ]:
daily_results[daily_results['team_id']==6]

# Step 3: SHAP
Explain why one team is at X% odds given their season stage.
That requires SHAP, which:
- decomposes odds into positive and negative drivers
- answers “most helpful” vs “most harmful”


#### Step 3B: Pick “one team at one date” snapshot

In [ ]:
team = "BOS"

team_row = (
    latest_snapshot
    .loc[latest_snapshot["tri_code"] == team]
    .iloc[0]
)

team_id = team_row["team_id"]
game_date = team_row["game_date"]
season_stage = team_row["season_bucket"]

print(f"Explaining {team} — {season_stage.upper()} season")


In [ ]:
row_mask = (
    (current_df["team_id"] == team_id) &
    (current_df["game_date"] == game_date)
)

assert row_mask.sum() == 1, "❌ Could not uniquely identify team row"

x_row = X_current.loc[row_mask]

assert x_row.shape == (1, len(FEATURE_COLS))
print("✅ Feature row selected for SHAP")


In [ ]:
shap_values = explainer.shap_values(x_row)

shap_series = (
    pd.Series(shap_values[0], index=FEATURE_COLS)
    .sort_values()
)


In [ ]:

most_harmful = shap_series.head(8)
most_helpful = shap_series.tail(8)

print("\n🔻 Most Harmful Drivers")
print(most_harmful.round(3))

print("\n🔺 Most Helpful Drivers")
print(most_helpful.round(3))


In [ ]:
baseline = explainer.expected_value
delta = shap_series.sum()
pred = pipeline.predict_proba(x_row)[0, 1]

print(f"Baseline: {baseline:.3f}")
print(f"Baseline + SHAP: {baseline + delta:.3f}")
print(f"Model prediction: {pred:.3f}")


In [ ]:
from IPython.display import HTML

HTML("""
<style>
.jp-Notebook {
    background-color: white !important;
}
</style>
""")

shap.initjs()

shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    x_row
)

In [ ]:
import matplotlib.pyplot as plt

# 🔹 Dark text
plt.rcParams.update({
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    x_row,
    matplotlib=True
)

# Let SHAP keep its light background
plt.show()
